# FitMe AI — сравнение моделей примерки (Kaggle, бесплатно)

**Что делает ноутбук**
1. Запускает **FASHN VTON 1.5** прямо на GPU Kaggle с разными настройками (число шагов, сегментация) и меряет время и видеопамять.
2. Через бесплатные Hugging Face Spaces прогоняет **IDM-VTON, Leffa, FitDiT, OOTDiffusion** на тех же фото.
3. Автоматически оценивает качество (точность цвета, сохранность лица и фона) и собирает сравнительные картинки.
4. Упаковывает всё в `fitme_benchmark_results.zip` (вкладка Output).

**Перед запуском**
- Settings → Accelerator → **GPU T4 x2** (или P100), Internet → **On**.
- Add-ons → Secrets → добавь `HF_TOKEN` (бесплатный токен с huggingface.co/settings/tokens, тип Read). Без него квота Spaces очень мала.
- Свои фото: загрузи датасет с папкой `test_set/` (фото + `pairs.csv` с колонками `person,garment,category`). Без него берётся демо-набор 3×3.

Потом **Run All**. Повторный запуск продолжает с того места, где кончилась квота.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q "fashn-vton @ git+https://github.com/fashn-AI/fashn-vton-1.5.git" httpx scikit-image
import os, sys, shutil, glob
WORK = "/kaggle/working/benchmark"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("ok")

In [ ]:
%%writefile bench_spaces.py
"""
FitMe — сравнение моделей виртуальной примерки через бесплатные Hugging Face Spaces.

Работает без своего GPU: модели считаются на ZeroGPU (бесплатно, лимит по времени в день).
Без токена квота ~3 минуты/день на IP, с бесплатным токеном HF — больше (см. README).

Примеры:
    python bench_spaces.py --demo                      # скачать демо-набор и прогнать все модели
    python bench_spaces.py --models fashn,idm          # только выбранные модели
    python bench_spaces.py --grids-only                # пересобрать сравнительные картинки

Повторный запуск пропускает уже готовые результаты, поэтому тест можно растянуть на несколько дней.
"""
from __future__ import annotations

import argparse
import csv
import io
import json
import os
import sys
import time
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import httpx
from PIL import Image, ImageDraw, ImageFont, ImageOps

ROOT = Path(__file__).resolve().parent
TEST_DIR = ROOT / "test_set"
RESULTS_DIR = ROOT / "results"
PAIRS_FILE = TEST_DIR / "pairs.csv"
RESULTS_CSV = RESULTS_DIR / "results.csv"

CATEGORIES = ("upper_body", "lower_body", "dresses")

DEMO_BASE = "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/example/"
DEMO_PEOPLE = ["human/00034_00.jpg", "human/00055_00.jpg", "human/01992_00.jpg"]
DEMO_GARMENTS = ["cloth/04469_00.jpg", "cloth/09163_00.jpg", "cloth/09263_00.jpg"]


class QuotaExceeded(RuntimeError):
    pass


# --------------------------------------------------------------------------- #
# Low-level Gradio HTTP client (works with Gradio 4.x, 5.x and 6.x servers)
# --------------------------------------------------------------------------- #
class Space:
    def __init__(self, space_id: str, token: str | None, timeout: float = 600.0):
        self.space_id = space_id
        self.host = "https://" + space_id.lower().replace("/", "-").replace(".", "-").replace("_", "-") + ".hf.space"
        headers = {"User-Agent": "FitMe-benchmark/1.0"}
        if token:
            headers["Authorization"] = f"Bearer {token}"
        self.http = httpx.Client(headers=headers, timeout=httpx.Timeout(30.0, read=timeout), follow_redirects=True)
        try:
            info = self.http.get(f"https://huggingface.co/api/spaces/{space_id}").json()
            self.host = (info.get("host") or self.host).rstrip("/")
            stage = (info.get("runtime") or {}).get("stage")
            if stage and stage != "RUNNING":
                raise RuntimeError(f"Space {space_id} сейчас {stage}")
        except httpx.HTTPError:
            pass
        cfg =self.http.get(self.host + "/config").raise_for_status().json()
        self.prefix = cfg.get("api_prefix") or ""
        if self.prefix == "/":
            self.prefix = ""
        self.fn_index = {d.get("api_name"): i for i, d in enumerate(cfg.get("dependencies", [])) if d.get("api_name")}

    def upload(self, path: Path, max_side: int = 1280) -> dict[str, Any]:
        image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
        image.thumbnail((max_side, max_side))
        buf = io.BytesIO()
        image.save(buf, format="JPEG", quality=95)
        r = self.http.post(self.host + self.prefix + "/upload", files={"files": (path.stem + ".jpg", buf.getvalue(), "image/jpeg")})
        r.raise_for_status()
        return {"path": r.json()[0], "meta": {"_type": "gradio.FileData"}}

    def call(self, api_name: str, data: list[Any]) -> tuple[list[Any], float | None]:
        api = api_name.lstrip("/")
        if api not in self.fn_index:
            raise RuntimeError(f"{self.space_id}: endpoint /{api} not found (Space changed?)")
        session = uuid.uuid4().hex[:11]
        join = self.http.post(
            self.host + self.prefix + "/queue/join",
            json={"data": data, "fn_index": self.fn_index[api], "session_hash": session, "event_data": None, "trigger_id": None},
        )
        join.raise_for_status()
        with self.http.stream("GET", self.host + self.prefix + "/queue/data", params={"session_hash": session}) as stream:
            for line in stream.iter_lines():
                if not line.startswith("data:"):
                    continue
                msg = json.loads(line[5:].strip())
                if msg.get("msg") != "process_completed":
                    continue
                out = msg.get("output") or {}
                if msg.get("success"):
                    return out.get("data") or [], out.get("duration")
                err = str(out.get("error") or msg.get("title") or "unknown error")
                if "quota" in err.lower():
                    raise QuotaExceeded(err)
                raise RuntimeError(err)
        raise RuntimeError("stream closed without result")

    def download(self, file_data: Any, dest: Path) -> None:
        while isinstance(file_data, list):  # Gallery -> first item
            file_data = file_data[0]
        if isinstance(file_data, dict) and "image" in file_data:  # Gallery item
            file_data = file_data["image"]
        url = file_data.get("url") if isinstance(file_data, dict) else None
        if not url:
            url = self.host + self.prefix + "/file=" + file_data["path"]
        r = self.http.get(url)
        r.raise_for_status()
        Image.open(io.BytesIO(r.content)).convert("RGB").save(dest, format="PNG")


def _strip(fd: Any) -> Any:
    if isinstance(fd, dict) and "path" in fd:
        return {"path": fd["path"], "meta": {"_type": "gradio.FileData"}}
    return fd


# --------------------------------------------------------------------------- #
# Model adapters
# --------------------------------------------------------------------------- #
@dataclass
class Model:
    key: str
    title: str
    space: str
    license: str
    categories: tuple[str, ...]
    run: Callable[[Space, Path, Path, str], tuple[Any, float | None]]


def run_fashn(s: Space, person: Path, garment: Path, category: str):
    cat = {"upper_body": "tops", "lower_body": "bottoms", "dresses": "one-pieces"}[category]
    data, dur = s.call("try_on", [s.upload(person), s.upload(garment), cat, "flat-lay", 30, 1.5, 42, True])
    return data[0], dur


def run_idm(s: Space, person: Path, garment: Path, category: str):
    p = s.upload(person)
    data, dur = s.call("tryon", [{"background": p, "layers": [], "composite": None}, s.upload(garment), "garment", True, True, 30, 42])
    return data[0], dur


def run_leffa(s: Space, person: Path, garment: Path, category: str):
    model_type = "viton_hd" if category == "upper_body" else "dress_code"
    data, dur = s.call("leffa_predict_vt", [s.upload(person), s.upload(garment), False, 30, 2.5, 42, model_type, category, False])
    return data[0], dur


def run_fitdit(s: Space, person: Path, garment: Path, category: str):
    cat = {"upper_body": "Upper-body", "lower_body": "Lower-body", "dresses": "Dresses"}[category]
    p = s.upload(person)
    mask, d1 = s.call("generate_mask", [p, cat, 0, 0, 0, 0])
    editor = mask[0]
    pre_mask = {
        "background": _strip(editor.get("background")),
        "layers": [_strip(x) for x in editor.get("layers") or []],
        "composite": _strip(editor.get("composite")),
        "id": None,
    }
    data, d2 = s.call("process", [p, s.upload(garment), pre_mask, _strip(mask[1]), 20, 2, 42, 1, "768x1024"])
    return data[0], (d1 or 0) + (d2 or 0)


def run_ootd(s: Space, person: Path, garment: Path, category: str):
    if category == "upper_body":
        data, dur = s.call("process_hd", [s.upload(person), s.upload(garment), 1, 20, 2, 42])
    else:
        cat = {"lower_body": "Lower-body", "dresses": "Dress"}[category]
        data, dur = s.call("process_dc", [s.upload(person), s.upload(garment), cat, 1, 20, 2, 42])
    return data[0], dur


MODELS: dict[str, Model] = {
    m.key: m
    for m in [
        Model("fashn", "FASHN VTON 1.5", "fashn-ai/fashn-vton-1.5", "Apache-2.0", CATEGORIES, run_fashn),
        Model("idm", "IDM-VTON", "yisol/IDM-VTON", "CC BY-NC-SA 4.0", ("upper_body",), run_idm),
        Model("leffa", "Leffa", "franciszzj/Leffa", "MIT (данные VITON-HD/DressCode)", CATEGORIES, run_leffa),
        Model("fitdit", "FitDiT", "BoyuanJiang/FitDiT", "CC BY-NC-SA 4.0", CATEGORIES, run_fitdit),
        Model("ootd", "OOTDiffusion", "levihsu/OOTDiffusion", "CC BY-NC-SA 4.0", CATEGORIES, run_ootd),
    ]
}


# --------------------------------------------------------------------------- #
# Test set, results, grids
# --------------------------------------------------------------------------- #
def download_demo() -> None:
    demo = TEST_DIR / "demo"
    demo.mkdir(parents=True, exist_ok=True)
    rows = []
    with httpx.Client(follow_redirects=True, timeout=60) as http:
        for rel in DEMO_PEOPLE + DEMO_GARMENTS:
            dest = demo / rel.replace("/", "_")
            if not dest.exists():
                dest.write_bytes(http.get(DEMO_BASE + rel).raise_for_status().content)
    for p in DEMO_PEOPLE:
        for g in DEMO_GARMENTS:
            rows.append({"person": f"demo/{p.replace('/', '_')}", "garment": f"demo/{g.replace('/', '_')}", "category": "upper_body"})
    write_pairs(rows)
    print(f"Демо-набор: {len(rows)} пар в {PAIRS_FILE}")


def write_pairs(rows: list[dict[str, str]]) -> None:
    PAIRS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with PAIRS_FILE.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["person", "garment", "category"])
        w.writeheader()
        w.writerows(rows)


def read_pairs() -> list[dict[str, str]]:
    if not PAIRS_FILE.exists():
        sys.exit(f"Нет {PAIRS_FILE}. Запусти с --demo или создай pairs.csv (person,garment,category).")
    pairs = []
    with PAIRS_FILE.open(encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            row = {k.strip(): (v or "").strip() for k, v in row.items()}
            if row.get("category") not in CATEGORIES:
                print(f"Пропуск: неизвестная категория {row}")
                continue
            for key in ("person", "garment"):
                if not (TEST_DIR / row[key]).exists():
                    sys.exit(f"Файл не найден: {TEST_DIR / row[key]}")
            pairs.append(row)
    return pairs


def pair_id(row: dict[str, str]) -> str:
    return f"{Path(row['person']).stem}__{Path(row['garment']).stem}"


def log_result(row: dict[str, Any]) -> None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    new = not RESULTS_CSV.exists()
    with RESULTS_CSV.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["time", "model", "pair", "category", "ok", "wall_sec", "gpu_sec", "error"])
        if new:
            w.writeheader()
        w.writerow(row)


def build_grids(pairs: list[dict[str, str]], models: list[str]) -> None:
    out_dir = RESULTS_DIR / "grids"
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        font = ImageFont.truetype("arial.ttf", 18)
    except OSError:
        font = ImageFont.load_default()
    W, H, pad = 300, 400, 34
    for row in pairs:
        pid = pair_id(row)
        cells = [("Человек", TEST_DIR / row["person"]), ("Вещь", TEST_DIR / row["garment"])]
        cells += [(MODELS[m].title if m in MODELS else m, RESULTS_DIR / m / f"{pid}.png") for m in models]
        if not any(p.exists() for _, p in cells[2:]):
            continue
        grid = Image.new("RGB", (W * len(cells), H + pad), "white")
        draw = ImageDraw.Draw(grid)
        for i, (label, path) in enumerate(cells):
            draw.text((i * W + 8, 8), label, fill="black", font=font)
            if path.exists():
                img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
                img.thumbnail((W, H))
                grid.paste(img, (i * W + (W - img.width) // 2, pad + (H - img.height) // 2))
            else:
                draw.text((i * W + 8, pad + H // 2), "нет результата", fill="gray", font=font)
        grid.save(out_dir / f"{pid}.jpg", quality=90)
    print(f"Сравнительные картинки: {out_dir}")


def write_rating_template(pairs: list[dict[str, str]], models: list[str]) -> None:
    path = RESULTS_DIR / "ratings.csv"
    if path.exists():
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["model", "pair", "вещь_похожа_1_5", "человек_узнаваем_1_5", "реализм_1_5", "посадка_1_5", "комментарий"])
        for row in pairs:
            for m in models:
                if (RESULTS_DIR / m / f"{pair_id(row)}.png").exists():
                    w.writerow([m, pair_id(row), "", "", "", "", ""])
    print(f"Шаблон ручной оценки: {path}")


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--demo", action="store_true", help="скачать демо-набор (3 человека x 3 вещи)")
    ap.add_argument("--models", default=",".join(MODELS), help="через запятую: " + ",".join(MODELS))
    ap.add_argument("--grids-only", action="store_true")
    ap.add_argument("--token", default=os.getenv("HF_TOKEN"), help="токен Hugging Face (или переменная HF_TOKEN)")
    args = ap.parse_args()

    models = [m.strip() for m in args.models.split(",") if m.strip()]
    unknown = [m for m in models if m not in MODELS]
    if unknown:
        sys.exit(f"Неизвестные модели: {unknown}. Доступны: {list(MODELS)}")

    if args.demo:
        download_demo()
    pairs = read_pairs()

    if not args.grids_only:
        if not args.token:
            print("ВНИМАНИЕ: без HF_TOKEN бесплатная квота GPU очень мала (Leffa не влезет вообще).")
        for key in models:
            model = MODELS[key]
            todo = [r for r in pairs if r["category"] in model.categories and not (RESULTS_DIR / key / f"{pair_id(r)}.png").exists()]
            if not todo:
                print(f"[{key}] всё уже готово")
                continue
            print(f"[{key}] {model.title}: {len(todo)} пар")
            try:
                space = Space(model.space, args.token)
            except Exception as exc:
                print(f"[{key}] Space недоступен: {exc}")
                continue
            (RESULTS_DIR / key).mkdir(parents=True, exist_ok=True)
            for row in todo:
                pid = pair_id(row)
                t0 = time.time()
                entry = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "model": key, "pair": pid, "category": row["category"]}
                try:
                    file_data, gpu = model.run(space, TEST_DIR / row["person"], TEST_DIR / row["garment"], row["category"])
                    space.download(file_data, RESULTS_DIR / key / f"{pid}.png")
                    entry.update(ok=1, wall_sec=round(time.time() - t0, 1), gpu_sec=round(gpu or 0, 1), error="")
                    print(f"  ok  {pid}  {entry['wall_sec']} c")
                    log_result(entry)
                except QuotaExceeded as exc:
                    print(f"  квота ZeroGPU закончилась для {key}: {exc}\n  Продолжи завтра — готовые результаты сохранены.")
                    break
                except Exception as exc:
                    entry.update(ok=0, wall_sec=round(time.time() - t0, 1), gpu_sec="", error=f"{type(exc).__name__}: {str(exc)[:300]}")
                    print(f"  ERR {pid}: {entry['error']}")
                    log_result(entry)

    all_dirs = sorted(p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and p.name != "grids") if RESULTS_DIR.exists() else []
    build_grids(pairs, all_dirs)
    write_rating_template(pairs, all_dirs)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluate.py
"""
FitMe — автоматическая оценка результатов примерки (без эталонных фото).

Для каждого результата считает:
  color_dE      — насколько цвет вещи на результате отличается от цвета на фото вещи
                  (ΔE в пространстве Lab, меньше = лучше; <10 почти незаметно, >25 — другой цвет)
  face_ssim     — насколько сохранено лицо и волосы (0..1, больше = лучше)
  bg_ssim       — насколько сохранён фон (0..1, больше = лучше)
  garment_cover — доля кадра, занятая вещью нужной категории (0 = модель вещь не нарисовала)

Разметку частей тела делает SegFormer (mattmdjaga/segformer_b2_clothes), работает и на CPU.

    python evaluate.py                # оценить всё в results/
    python evaluate.py --device cuda
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parent
TEST_DIR = ROOT / "test_set"
RESULTS_DIR = ROOT / "results"

# Классы segformer_b2_clothes
BG, HAIR, FACE = 0, 2, 11
UPPER, SKIRT, PANTS, DRESS = 4, 5, 6, 7
CATEGORY_CLASSES = {"upper_body": [UPPER], "lower_body": [SKIRT, PANTS], "dresses": [DRESS, UPPER, SKIRT]}


# ----------------------------- colour helpers ------------------------------ #
def rgb_to_lab(rgb: np.ndarray) -> np.ndarray:
    """rgb: (..., 3) uint8 -> Lab (D65)."""
    c = rgb.astype(np.float64) / 255.0
    c = np.where(c > 0.04045, ((c + 0.055) / 1.055) ** 2.4, c / 12.92)
    m = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    xyz = c @ m.T / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.stack([116 * f[..., 1] - 16, 500 * (f[..., 0] - f[..., 1]), 200 * (f[..., 1] - f[..., 2])], axis=-1)


def garment_pixels_flatlay(img: np.ndarray) -> np.ndarray:
    """Пиксели вещи на фото товара: всё, что заметно отличается от цвета углов (фона)."""
    h, w, _ = img.shape
    s = max(4, min(h, w) // 24)
    corners = np.concatenate([img[:s, :s], img[:s, -s:], img[-s:, :s], img[-s:, -s:]]).reshape(-1, 3)
    bg = np.median(corners, axis=0)
    diff = np.abs(img.astype(int) - bg).max(axis=-1)
    return img[diff > 30]


def median_lab(pixels: np.ndarray) -> np.ndarray | None:
    if len(pixels) < 200:
        return None
    return np.median(rgb_to_lab(pixels.reshape(-1, 3)), axis=0)


def delta_e(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


# ------------------------------ similarity --------------------------------- #
def masked_ssim(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> float | None:
    if mask.sum() < 400:
        return None
    from skimage.metrics import structural_similarity

    ys, xs = np.where(mask)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    ga = np.asarray(Image.fromarray(a[y0:y1, x0:x1]).convert("L"), dtype=np.float64)
    gb = np.asarray(Image.fromarray(b[y0:y1, x0:x1]).convert("L"), dtype=np.float64)
    if min(ga.shape) < 7:
        return None
    _, full = structural_similarity(ga, gb, data_range=255, full=True)
    return float(full[mask[y0:y1, x0:x1]].mean())


# ------------------------------- parsing ----------------------------------- #
class Parser:
    def __init__(self, device: str = "cpu"):
        import torch
        from transformers import AutoModelForSemanticSegmentation, SegformerImageProcessor

        self.torch = torch
        self.device = device
        name = "mattmdjaga/segformer_b2_clothes"
        self.proc = SegformerImageProcessor.from_pretrained(name)
        self.model = AutoModelForSemanticSegmentation.from_pretrained(name).to(device).eval()

    def __call__(self, img: Image.Image) -> np.ndarray:
        inputs = self.proc(images=img, return_tensors="pt").to(self.device)
        with self.torch.no_grad():
            logits = self.model(**inputs).logits
        up = self.torch.nn.functional.interpolate(logits, size=img.size[::-1], mode="bilinear", align_corners=False)
        return up.argmax(dim=1)[0].cpu().numpy()


def load(path: Path, size: tuple[int, int] | None = None) -> Image.Image:
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    if size and img.size != size:
        img = img.resize(size, Image.Resampling.LANCZOS)
    return img


def evaluate_one(parser: Parser, person: Path, garment: Path, result: Path, category: str) -> dict:
    src = load(person)
    src.thumbnail((768, 1024))
    out = load(result, src.size)
    a, b = np.asarray(src), np.asarray(out)
    seg_a, seg_b = parser(src), parser(out)

    garment_mask = np.isin(seg_b, CATEGORY_CLASSES[category])
    target = median_lab(garment_pixels_flatlay(np.asarray(load(garment))))
    got = median_lab(b[garment_mask])
    face = np.isin(seg_a, [FACE, HAIR]) & np.isin(seg_b, [FACE, HAIR])
    bg = (seg_a == BG) & (seg_b == BG)

    return {
        "color_dE": round(delta_e(target, got), 1) if target is not None and got is not None else "",
        "face_ssim": _r(masked_ssim(a, b, face)),
        "bg_ssim": _r(masked_ssim(a, b, bg)),
        "garment_cover": round(float(garment_mask.mean()), 3),
    }


def _r(v: float | None) -> str | float:
    return "" if v is None else round(v, 3)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--device", default="cpu")
    args = ap.parse_args()

    pairs = list(csv.DictReader((TEST_DIR / "pairs.csv").open(encoding="utf-8-sig")))
    by_id = {f"{Path(r['person']).stem}__{Path(r['garment']).stem}": r for r in pairs}
    parser = Parser(args.device)

    rows = []
    for model_dir in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir() and p.name != "grids"):
        for res in sorted(model_dir.glob("*.png")):
            pair = by_id.get(res.stem)
            if not pair:
                continue
            try:
                m = evaluate_one(parser, TEST_DIR / pair["person"], TEST_DIR / pair["garment"], res, pair["category"])
            except Exception as exc:  # одна битая картинка не должна ронять всю оценку
                print(f"skip {model_dir.name}/{res.name}: {exc}")
                continue
            rows.append({"model": model_dir.name, "pair": res.stem, **m})
            print(model_dir.name, res.stem, m)

    if not rows:
        print("Нет результатов для оценки.")
        return
    out = RESULTS_DIR / "metrics.csv"
    with out.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    print("\nСредние по моделям (color_dE ↓, face_ssim ↑, bg_ssim ↑):")
    summary = []
    for model in sorted({r["model"] for r in rows}):
        sel = [r for r in rows if r["model"] == model]

        def avg(key: str) -> str:
            vals = [float(r[key]) for r in sel if r[key] != ""]
            return f"{np.mean(vals):.3f}" if vals else "-"

        summary.append({"model": model, "n": len(sel), "color_dE": avg("color_dE"), "face_ssim": avg("face_ssim"), "bg_ssim": avg("bg_ssim"), "garment_cover": avg("garment_cover")})
        print(f"  {model:22s} n={len(sel):3d}  dE={summary[-1]['color_dE']:>7}  face={summary[-1]['face_ssim']:>6}  bg={summary[-1]['bg_ssim']:>6}")
    with (RESULTS_DIR / "summary.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(summary[0]))
        w.writeheader()
        w.writerows(summary)
    print(f"\nПодробно: {out}\nИтог: {RESULTS_DIR / 'summary.csv'}")


if __name__ == "__main__":
    main()

## 1. Тестовый набор

In [ ]:
user_sets = glob.glob("/kaggle/input/**/test_set/pairs.csv", recursive=True)
if user_sets:
    src = os.path.dirname(user_sets[0])
    shutil.copytree(src, "test_set", dirs_exist_ok=True)
    print("Свой набор:", src)
else:
    !python bench_spaces.py --demo --grids-only
!cat test_set/pairs.csv

## 2. FASHN VTON 1.5 локально на GPU Kaggle — подбор настроек

In [ ]:
import csv, time, torch
from pathlib import Path
from PIL import Image, ImageOps
from huggingface_hub import hf_hub_download
from fashn_vton import TryOnPipeline

W = "/kaggle/working/fitme_weights"
os.makedirs(W + "/dwpose", exist_ok=True)
hf_hub_download("fashn-ai/fashn-vton-1.5", "model.safetensors", local_dir=W)
for f in ("yolox_l.onnx", "dw-ll_ucoco_384.onnx"):
    hf_hub_download("fashn-ai/DWPose", f, local_dir=W + "/dwpose")
pipe = TryOnPipeline(weights_dir=W, device="cuda")

CAT = {"upper_body": "tops", "lower_body": "bottoms", "dresses": "one-pieces"}
CONFIGS = [  # имя папки, шаги, segmentation_free
    ("fashn_local_s10", 10, True),
    ("fashn_local_s20", 20, True),
    ("fashn_local_s30", 30, True),
    ("fashn_local_s50", 50, True),
    ("fashn_local_s30_seg", 30, False),
]
pairs = list(csv.DictReader(open("test_set/pairs.csv", encoding="utf-8-sig")))
timing = []
for name, steps, seg_free in CONFIGS:
    out_dir = Path("results") / name
    out_dir.mkdir(parents=True, exist_ok=True)
    for row in pairs:
        pid = f"{Path(row['person']).stem}__{Path(row['garment']).stem}"
        dest = out_dir / f"{pid}.png"
        if dest.exists():
            continue
        person = ImageOps.exif_transpose(Image.open("test_set/" + row["person"])).convert("RGB")
        garment = ImageOps.exif_transpose(Image.open("test_set/" + row["garment"])).convert("RGB")
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
        try:
            res = pipe(person_image=person, garment_image=garment, category=CAT[row["category"]],
                       garment_photo_type="flat-lay", num_samples=1, num_timesteps=steps,
                       guidance_scale=1.5, seed=42, segmentation_free=seg_free)
            torch.cuda.synchronize()
            res.images[0].save(dest)
            timing.append({"config": name, "pair": pid, "sec": round(time.time() - t0, 2),
                           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2), "error": ""})
        except Exception as exc:
            timing.append({"config": name, "pair": pid, "sec": "", "peak_vram_gb": "", "error": repr(exc)[:300]})
        print(timing[-1])
if timing:
    new = not Path("results/local_timing.csv").exists()
    with open("results/local_timing.csv", "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(timing[0]))
        if new: w.writeheader()
        w.writerows(timing)
del pipe; torch.cuda.empty_cache()

## 3. Другие модели через бесплатные Hugging Face Spaces

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN найден")
except Exception:
    print("HF_TOKEN не задан — квота будет маленькой")
!python bench_spaces.py --models idm,leffa,fitdit,ootd,fashn

## 4. Автоматическая оценка и итоговая таблица

In [ ]:
!python evaluate.py --device cuda
import pandas as pd
summary = pd.read_csv("results/summary.csv")
if os.path.exists("results/local_timing.csv"):
    t = pd.read_csv("results/local_timing.csv").groupby("config")[["sec", "peak_vram_gb"]].mean().round(2)
    summary = summary.merge(t, left_on="model", right_index=True, how="left")
if os.path.exists("results/results.csv"):
    s = pd.read_csv("results/results.csv")
    s = s[s.ok == 1].groupby("model")[["wall_sec", "gpu_sec"]].mean().round(1)
    summary = summary.merge(s, left_on="model", right_index=True, how="left")
summary.to_csv("results/summary_full.csv", index=False)
summary

In [ ]:
from IPython.display import Image as Show, display
for g in sorted(glob.glob("results/grids/*.jpg"))[:9]:
    display(Show(g, width=1400))
shutil.make_archive("/kaggle/working/fitme_benchmark_results", "zip", "results")
print("Готово: /kaggle/working/fitme_benchmark_results.zip — скачай во вкладке Output")